In [1]:
import numpy as np # linear algebra
import pandas as pd
import sqlite3

# Load Dataset

In [2]:
# Establish connection to database
db_path = r"C:\Repositories\DI-Bootcamp\Week19\DailyChallenge\database.sqlite"
conn = sqlite3.connect(db_path)

In [3]:
# Load the master table to understand the structure
query = '''
SELECT name
FROM sqlite_master
WHERE type='table'; 
'''

tables = pd.read_sql(query, conn)

# Convert Table Names to a List for future analysis
table_names = tables["name"].tolist()
table_names

['Player',
 'Extra_Runs',
 'Batsman_Scored',
 'Batting_Style',
 'Bowling_Style',
 'Country',
 'Season',
 'City',
 'Outcome',
 'Win_By',
 'Wicket_Taken',
 'Venue',
 'Extra_Type',
 'Out_Type',
 'Toss_Decision',
 'Umpire',
 'Team',
 'Ball_by_Ball',
 'sysdiagrams',
 'sqlite_sequence',
 'Match',
 'Rolee',
 'Player_Match']

In [44]:
# Load each table and print column names
table_columns = {}

for table in table_names:
    df = pd.read_sql(f"SELECT * FROM {table} LIMIT 1;", conn)
    table_columns[table] = df.columns.tolist()

for k, v in table_columns.items():
    print(f"{k}:")
    print(', '.join(c for c in v), '\n')

Player:
Player_Id, Player_Name, DOB, Batting_hand, Bowling_skill, Country_Name 

Extra_Runs:
Match_Id, Over_Id, Ball_Id, Extra_Type_Id, Extra_Runs, Innings_No 

Batsman_Scored:
Match_Id, Over_Id, Ball_Id, Runs_Scored, Innings_No 

Batting_Style:
Batting_Id, Batting_hand 

Bowling_Style:
Bowling_Id, Bowling_skill 

Country:
Country_Id, Country_Name 

Season:
Season_Id, Man_of_the_Series, Orange_Cap, Purple_Cap, Season_Year 

City:
City_Id, City_Name, Country_id 

Outcome:
Outcome_Id, Outcome_Type 

Win_By:
Win_Id, Win_Type 

Wicket_Taken:
Match_Id, Over_Id, Ball_Id, Player_Out, Kind_Out, Fielders, Innings_No 

Venue:
Venue_Id, Venue_Name, City_Id 

Extra_Type:
Extra_Id, Extra_Name 

Out_Type:
Out_Id, Out_Name 

Toss_Decision:
Toss_Id, Toss_Name 

Umpire:
Umpire_Id, Umpire_Name, Umpire_Country 

Team:
Team_Id, Team_Name 

Ball_by_Ball:
Match_Id, Over_Id, Ball_Id, Innings_No, Team_Batting, Team_Bowling, Striker_Batting_Position, Striker, Non_Striker, Bowler 

sysdiagrams:
name, principal_

Query 1: Select All Columns from Player’s Table     

In [50]:
# Write and execute a SQL query to select all columns from the Player_Match table
query = '''
SELECT *
FROM Player p 
JOIN Player_Match m ON p.Player_Id = m.Player_Id
LIMIT 1 
'''

# Load Query as DF
df = pd.read_sql(query, conn)

# Print Columns
cols = df.columns.tolist()
print(set(cols))

{'Player_Name', 'DOB', 'Role_Id', 'Bowling_skill', 'Team_Id', 'Player_Id', 'Batting_hand', 'Match_Id', 'Country_Name'}


Query 2: Batsman vs Runs

In [56]:
query = '''
SELECT
    p.Player_Id AS player_id,
    p.Player_Name AS name,
    SUM(bs.Runs_Scored) AS total_runs
FROM Batsman_Scored bs
JOIN Ball_by_Ball bb ON bs.Match_Id = bb.Match_Id
                    AND bs.Over_Id = bb.Over_Id 
                    AND bs.Ball_Id = bb.Ball_Id  
                    AND bs.Innings_No = bb.Innings_No  -- MUST JOIN on Match_Id, Innings_No, Over_Id, and Ball because Ball_Id is not globally unique
JOIN Player p ON bb.Striker = p.Player_Id              -- Striker, Non-Striker, Bowling are numbers that represent a player's ID (player_id)
GROUP BY p.Player_Id, p.Player_Name
'''

df = pd.read_sql(query, conn)
print(df)

     player_id             name  total_runs
0            1       SC Ganguly        1349
1            2      BB McCullum        2435
2            3       RT Ponting          91
3            4        DJ Hussey        1322
4            5  Mohammad Hafeez          64
..         ...              ...         ...
429        430          A Zampa           0
430        431           N Rana         104
431        432        S Kaushik           0
432        433       ER Dwivedi          24
433        434        CJ Jordan           3

[434 rows x 3 columns]


Query 3: Fifties and Hundreds

In [46]:
# Write and execute a SQL query to calculate the number of fifties and hundreds scored by each batsman
query = '''
WITH runs_inning AS (
    SELECT
        p.Player_Id AS id,
        p.Player_Name AS player,
        bb.Match_Id AS match,
        bb.Innings_No AS inning,
        m.Season_Id AS season,
        SUM(COALESCE(bs.Runs_Scored, 0)) AS runs
    FROM Ball_by_Ball bb
    LEFT JOIN Batsman_Scored bs
        ON bb.Match_Id = bs.Match_Id
       AND bb.Over_Id = bs.Over_Id
       AND bb.Ball_Id = bs.Ball_Id
       AND bb.Innings_No = bs.Innings_No
    JOIN Match m
        ON bb.Match_Id = m.Match_Id
    JOIN Player p
        ON bb.Striker = p.Player_Id
    GROUP BY
        id, player, match, inning, season
)
SELECT
    id,
    player,
    SUM(CASE WHEN runs BETWEEN 50 AND 99 THEN 1 ELSE 0 END) AS fifties,
    SUM(CASE WHEN runs >= 100 THEN 1 ELSE 0 END) AS hundreds
FROM runs_inning
GROUP BY id, player
ORDER BY hundreds DESC, 
        fifties DESC; 
'''

df = pd.read_sql(query, conn)

print(df)

      id           player  fifties  hundreds
0    162         CH Gayle       20         5
1      8          V Kohli       26         4
2    110   AB de Villiers       21         3
3    187        DA Warner       32         2
4     41         V Sehwag       16         2
..   ...              ...      ...       ...
429   14          P Kumar        0         0
430   13        AA Noffke        0         0
431   12          B Akhil        0         0
432    5  Mohammad Hafeez        0         0
433    3       RT Ponting        0         0

[434 rows x 4 columns]


Query 4: Best Bowling Figures

In [45]:
# Write and execute a SQL query to find the best bowling figures for each bowler
query = '''
WITH bowler_stats AS (
    SELECT
        bb.Bowler AS Bowler_Id,
        p.Player_Name AS Bowler_Name,
        bb.Match_Id,
        SUM(bs.Runs_Scored) 
            + COALESCE(SUM(er.Extra_Runs), 0) AS Runs_Conceded,
        COUNT(wt.Player_Out) AS Wickets
    FROM Ball_by_Ball bb
    LEFT JOIN Batsman_Scored bs
        ON bb.Match_Id = bs.Match_Id
       AND bb.Over_Id = bs.Over_Id
       AND bb.Ball_Id = bs.Ball_Id
       AND bb.Innings_No = bs.Innings_No
    LEFT JOIN Extra_Runs er
        ON bb.Match_Id = er.Match_Id
       AND bb.Over_Id = er.Over_Id
       AND bb.Ball_Id = er.Ball_Id
       AND bb.Innings_No = er.Innings_No
    LEFT JOIN Wicket_Taken wt
        ON bb.Match_Id = wt.Match_Id
       AND bb.Over_Id = wt.Over_Id
       AND bb.Ball_Id = wt.Ball_Id
       AND bb.Innings_No = wt.Innings_No
       AND wt.Kind_Out != 'run out'
    JOIN Player p
        ON bb.Bowler = p.Player_Id
    GROUP BY bb.Bowler, p.Player_Name, bb.Match_Id
),
best_figures AS (
    SELECT Bowler_Id, Bowler_Name, Match_Id, Runs_Conceded, Wickets,
           ROW_NUMBER() OVER (
               PARTITION BY Bowler_Id
               ORDER BY Wickets DESC, Runs_Conceded ASC
           ) AS rn
    FROM bowler_stats
)
SELECT Bowler_Id, Bowler_Name, Match_Id, Wickets, Runs_Conceded
FROM best_figures
WHERE rn = 1
ORDER BY Wickets DESC, Runs_Conceded ASC;
'''

df = pd.read_sql(query, conn)

print(df)

     Bowler_Id    Bowler_Name  Match_Id  Wickets  Runs_Conceded
0          102  Sohail Tanvir    336010        6             15
1          430        A Zampa    980984        6             19
2          362      DJG Sammy    598061        6             23
3          334     AD Russell    980968        6             25
4          124       A Kumble    392187        5              6
..         ...            ...       ...      ...            ...
326        444      MB Parmar    419140        0             33
327        449       RW Price    501272        0             33
328        317     RR Bhatkal    548323        0             35
329        337    Sunny Gupta    548385        0             47
330        454       MG Neser    598069        0             62

[331 rows x 5 columns]


Query 5: Comprehensive Career Metrics

In [59]:
query = '''
WITH 
-- Total runs
total_runs AS (
    SELECT
        p.Player_Id AS player_id,
        p.Player_Name AS name,
        SUM(bs.Runs_Scored) AS total_runs
    FROM Batsman_Scored bs
    JOIN Ball_by_Ball bb
        ON bs.Match_Id = bb.Match_Id
       AND bs.Over_Id = bb.Over_Id
       AND bs.Ball_Id = bb.Ball_Id
       AND bs.Innings_No = bb.Innings_No
    JOIN Player p
        ON bb.Striker = p.Player_Id
    GROUP BY p.Player_Id, p.Player_Name
),

-- Runs per innings for fifties/hundreds
runs_inning AS (
    SELECT
        p.Player_Id AS player_id,
        p.Player_Name AS player,
        bb.Match_Id AS match,
        bb.Innings_No AS inning,
        SUM(COALESCE(bs.Runs_Scored, 0)) AS runs
    FROM Ball_by_Ball bb
    LEFT JOIN Batsman_Scored bs
        ON bb.Match_Id = bs.Match_Id
       AND bb.Over_Id = bs.Over_Id
       AND bb.Ball_Id = bs.Ball_Id
       AND bb.Innings_No = bs.Innings_No
    JOIN Player p
        ON bb.Striker = p.Player_Id
    GROUP BY player_id, player, match, inning
),

fifties_hundreds AS (
    SELECT
        player_id,
        SUM(CASE WHEN runs BETWEEN 50 AND 99 THEN 1 ELSE 0 END) AS fifties,
        SUM(CASE WHEN runs >= 100 THEN 1 ELSE 0 END) AS hundreds
    FROM runs_inning
    GROUP BY player_id
),

-- Bowling stats per match
bowler_stats AS (
    SELECT
        bb.Bowler AS player_id,
        SUM(bs.Runs_Scored) + COALESCE(SUM(er.Extra_Runs), 0) AS runs_conceded,
        COUNT(wt.Player_Out) AS wickets,
        bb.Match_Id
    FROM Ball_by_Ball bb
    LEFT JOIN Batsman_Scored bs
        ON bb.Match_Id = bs.Match_Id
       AND bb.Over_Id = bs.Over_Id
       AND bb.Ball_Id = bs.Ball_Id
       AND bb.Innings_No = bs.Innings_No
    LEFT JOIN Extra_Runs er
        ON bb.Match_Id = er.Match_Id
       AND bb.Over_Id = er.Over_Id
       AND bb.Ball_Id = er.Ball_Id
       AND bb.Innings_No = er.Innings_No
    LEFT JOIN Wicket_Taken wt
        ON bb.Match_Id = wt.Match_Id
       AND bb.Over_Id = wt.Over_Id
       AND bb.Ball_Id = wt.Ball_Id
       AND bb.Innings_No = wt.Innings_No
       AND wt.Kind_Out != 'run out'
    GROUP BY bb.Bowler, bb.Match_Id
),

-- Best bowling figures per player
best_bowling AS (
    SELECT player_id, wickets AS best_wickets, runs_conceded AS best_runs_conceded
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY wickets DESC, runs_conceded ASC) AS rn
        FROM bowler_stats
    ) sub
    WHERE rn = 1
)

-- Combine everything
SELECT
    t.player_id,
    t.name,
    t.total_runs,
    fh.fifties,
    fh.hundreds,
    bb.best_wickets,
    bb.best_runs_conceded
FROM total_runs t
LEFT JOIN fifties_hundreds fh
    ON t.player_id = fh.player_id
LEFT JOIN best_bowling bb
    ON t.player_id = bb.player_id
'''

df = pd.read_sql(query, conn)

print(df)

     player_id             name  total_runs  fifties  hundreds  best_wickets  \
0            1       SC Ganguly        1349        7         0           3.0   
1            2      BB McCullum        2435       11         2           NaN   
2            3       RT Ponting          91        0         0           NaN   
3            4        DJ Hussey        1322        5         0           2.0   
4            5  Mohammad Hafeez          64        0         0           1.0   
..         ...              ...         ...      ...       ...           ...   
429        430          A Zampa           0        0         0           6.0   
430        431           N Rana         104        1         0           0.0   
431        432        S Kaushik           0        0         0           3.0   
432        433       ER Dwivedi          24        0         0           NaN   
433        434        CJ Jordan           3        0         0           4.0   

     best_runs_conceded  
0            